In [4]:
import numpy as np
import cv2
from scipy.io import savemat

In [5]:
def image_to_cone(image):
    """
    Spectral fits and parameters for this display, measured by Barry and Hans
    Image has to be BGR!
    """
    #Raw RGB values
    im_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    red  = np.double(im_rgb[:,:,0])
    green  = np.double(im_rgb[:,:,1])
    blue  = np.double(im_rgb[:,:,2])
    
    #Hans' gamma correction
    rgamma = 0.01451  + 0.9855*((1.0*red/256)**2.3122)
    ggamma = 0.005123 + 0.9949*((1.0*green/256)**2.2752)
    bgamma = 0.02612  + 0.9739*((1.0*blue/256)**2.2818)
    
    lcone = 1.0*(2.74*rgamma + 3.4*ggamma + 1.34*bgamma)
    mcone = 1.21*(1.06*rgamma + 3.58*ggamma + 2.07*bgamma)
    scone = 0.212*rgamma + 8.28*ggamma + 285*bgamma

    return scone, mcone, lcone

In [8]:
vidcap = cv2.VideoCapture('./1x10_256.mpg')
counter = 0
success = True
m = np.zeros((256,256,9750))
l = np.zeros((256,256,9750))
rawimages = np.zeros((9750, 256, 256, 3), dtype='uint8')

while success and (counter < 9750):
    success, image = vidcap.read()
    scone, mcone, lcone = image_to_cone(image)
    m[:,:,counter] = mcone
    l[:,:,counter] = lcone
    rawimages[counter] = image
    counter += 1
    
mdic = {"m_cone": m[20:236,20:236,:]}
savemat("matlab_flowershow_m.mat", mdic)
ldic = {"l_cone": l[20:236,20:236,:]}
savemat("matlab_flowershow_l.mat", ldic)

rawdic = {"raw_image": rawimages[:,20:236,20:236,:]}
savemat("matlab_flowershow_rawdic.mat", rawdic)